## Imports and Installations

In [ ]:
import os
import sys

In [ ]:
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [ ]:
sys.path.insert(0, f'{project_dir}/code/Packages')

In [ ]:
import pandas as pd
import json
import preprocessing_nopos_gaz as preprocessing
from tqdm import tqdm
from glob import glob

Running in cloud using directory /content/drive/Shareddrives/.
Export directory set to /content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/NetSci Team/final_edgelists.


In [ ]:
preprocessing.set_is_local(False)
preprocessing.set_export_folder("final_edgelists")

nodelist = preprocessing.get_inverted_nodelist()

## Load Semsets

In [ ]:
directory = f'{project_dir}/data/7 - Semset Creation/semsets_leiden_mod_removed0.3.json'

with open(directory, 'r') as semset_file:
  semset_dict = json.load(semset_file)

## Load Senses

In [ ]:
directory = f'{project_dir}/data/6 - Sense Creation/sense_inventory'

filenames = glob(f'{directory}/*.json')
files = []
for file in filenames:
  files.append(pd.read_json(file, orient='index').transpose())

senses_df = pd.concat(files).sort_values('sense_id').reset_index(drop=True)

display(senses_df)

## Add Semset ID to Senses

In [ ]:
def get_semset_id(sense_id):
  for index in semset_dict:
    if sense_id in semset_dict[index]:
      return index
  
  return -1

In [ ]:
directory = f'{project_dir}/data/4 - Word Sense Induction/'
algorithm = 'leiden_mod'

senses_df['semset_id'] = None

for row in senses_df.iterrows(): # iterate through all senses
  
  print(row[1]['sense_id'])
  
  word = row[1]['word']

  with open(f'{directory}communities/filtered_10/{word}_{algorithm}.json') as f: # open communities
    comms = json.load(f)

  sense_id = row[1]['sense_id']
  comm_index = sense_id.split('_')[-1] # get community index
  
  if set(row[1]['community']) == set([nodelist[i] for i in comms['community'][comm_index]]): # double checks if the communities are the same
    semset_id = get_semset_id(row[1]['sense_id'][3:])

  else:
    print("Index doesn't match community.")
    semset_id = -1

  senses_df.iloc[row[0]]['semset_id'] = semset_id

display(senses_df)

In [ ]:
senses_df[senses_df['semset_id'] == -1] # check if all senses have a semset

In [ ]:
senses_df = senses_df[['sense_id', 'word', 'semset_id', 'community', 'example_sentences', 'contextual_info']] #reorder columns

In [ ]:
display(senses_df)

## Save Wordnet Entries

In [ ]:
for row in senses_df.iterrows():
  sense_id = row[1]['sense_id']

  # TODO: Update the destination folder
  directory = f'{project_dir}/data/8 - Wordnet Creation/wordnet_entries' 

  if not os.path.exists(directory):
    os.makedirs(directory)
  
  print(row[1]['word'])
  row[1].to_json(f'{directory}/{sense_id}.json', orient='index')  